In [ ]:
# M*xs + C = 0 form
import numpy as np
from numba import jit

## M * dx + C = 0

@jit
def Mfcn(t,xs,ps): # mass matrix for dynamics
    σ1, σ2, σ3, σr, σg, ψ, ϑ, φ, r, γ, xG, yG = xs
    g, R, mw, JW1, JW2, mr, JR, BR, h, mp, JPx, JPy, JPz, BP = ps
    sin,cos,tan=np.sin,np.cos,np.tan
    M = np.zeros((5,5))

    M[0,0] = JPz + JR + JW2 + R**2*mp + R**2*mw + 2*R*h*mp*cos(γ) + mr*r**2 + (JPx - JPz + h**2*mp)*cos(γ)**2
    M[0,2] = -R*h*mp*sin(γ) + (-JPx + JPz - h**2*mp)*sin(γ)*cos(γ)
    M[2,0] = M[0,2]
    M[1,1] = JW1 + R**2*mp + R**2*mr + R**2*mw
    M[1,2] = -R*mr*r
    M[2,1] = M[1,2]
    M[1,4] = R*h*mp*cos(γ)
    M[4,1] = M[1,4]
    M[2,2] = JPz + JR + JW2 + mr*r**2 + (JPx - JPz + h**2*mp)*sin(γ)**2
    M[3,3] = mr
    M[4,4] = JPy + h**2*mp
    return M

@jit
def Cfcn(t,xs,ps,us): # dynamics including all forces (control as well) + kinematics 
    σ1, σ2, σ3, σr, σg, ψ, ϑ, φ, r, γ, xG, yG = xs
    g, R, mw, JW1, JW2, mr, JR, BR, h, mp, JPx, JPy, JPz, BP = ps
    sin,cos,tan=np.sin,np.cos,np.tan
    F1, M2 = us
    C = np.empty(12)

    # dynamics part
    C[0] = BR*R**2*σ1 + BR*R*σr + F1*R - R*g*(mp + mw)*sin(ϑ) + R*mr*r*σ1**2 - g*h*mp*sin(ϑ)*cos(γ) + g*mr*r*cos(ϑ) + 2*mr*r*σ1*σr + σ1*σ3*(R*h*mp*sin(γ)*tan(ϑ) + (JPx - JPz + h**2*mp)*sin(γ)*cos(γ)*tan(ϑ)) + σ1*σg*(-2*R*h*mp*sin(γ) - 2*(JPx - JPz + h**2*mp)*sin(γ)*cos(γ)) + σ2*σ3*(-JW1 - R**2*mp - R**2*mw - R*h*mp*cos(γ) - R*mr*r*tan(ϑ)) + σ3**2*(R*h*mp*cos(γ)*tan(ϑ) + mr*r**2*tan(ϑ) + (JPx - JPz + h**2*mp)*cos(γ)**2*tan(ϑ) + (JPz + JR + JW2)*tan(ϑ)) + σ3*σg*(JPx - JPy - JPz - 2*R*h*mp*cos(γ) - 2*(JPx - JPz + h**2*mp)*cos(γ)**2)
    C[1] = BP*σ2 - BP*σg - M2 - R*h*mp*σ3**2*sin(γ) - R*h*mp*σg**2*sin(γ) - 2*R*mr*σ3*σr + σ1*σ3*(R**2*(mp - mr + mw) + R*h*mp*cos(γ) + R*mr*r*tan(ϑ))
    C[2] = JW1*σ1*σ2 + R*h*mp*σ2*σ3*sin(γ) + g*h*mp*sin(γ)*sin(ϑ) + 2*mr*r*σ3*σr + σ1*σ3*(R*mr*r - mr*r**2*tan(ϑ) + (-JPx + JPz - h**2*mp)*sin(γ)**2*tan(ϑ) + (-JPz - JR - JW2)*tan(ϑ)) + σ1*σg*(-JPx + JPy + JPz + 2*(JPx - JPz + h**2*mp)*sin(γ)**2) + σ3**2*(-JPx + JPz - h**2*mp)*sin(γ)*cos(γ)*tan(ϑ) + 2*σ3*σg*(JPx - JPz + h**2*mp)*sin(γ)*cos(γ)
    C[3] = BR*R*σ1 + BR*σr + F1 + R*mr*σ2*σ3 + g*mr*sin(ϑ) - mr*r*σ1**2 - mr*r*σ3**2
    C[4] = -BP*σ2 + BP*σg + M2 + R*h*mp*σ2*σ3*sin(γ)*tan(ϑ) - g*h*mp*sin(γ)*cos(ϑ) + σ1**2*(R*h*mp*sin(γ) + (JPx - JPz + h**2*mp)*sin(γ)*cos(γ)) + σ1*σ3*(JPx - JPz + R*h*mp*cos(γ) + h**2*mp - 2*(JPx - JPz + h**2*mp)*sin(γ)**2) + σ3**2*(-JPx + JPz - h**2*mp)*sin(γ)*cos(γ)

    #kinematics
    C[5]  = -σ3/cos(ϑ)
    C[6]  = -σ1
    C[7]  = -σ2 + σ3*tan(ϑ)
    C[8]  = -R*σ1 - σr
    C[9]  = σ3*tan(ϑ) - σg
    C[10] = -R*σ1*sin(ψ)*cos(ϑ) - R*σ2*cos(ψ)
    C[11] = R*σ1*cos(ψ)*cos(ϑ) - R*σ2*sin(ψ)

    return C

@jit
def control(t,xs,ps): 
    σ1, σ2, σ3, σr, σg, ψ, ϑ, φ, r, γ, xG, yG = xs
    g, R, mw, JW1, JW2, mr, JR, BR, h, mp, JPx, JPy, JPz, BP = ps
    sin,cos,tan=np.sin,np.cos,np.tan

    F1 = 0 # lateral controller
    M2 = 0 # logitudinal controller
    return np.array([F1,M2])

@jit
def dqs(xs,ps):
    σ1, σ2, σ3, σr, σg, ψ, ϑ, φ, r, γ, xG, yG = xs
    g, R, mw, JW1, JW2, mr, JR, BR, h, mp, JPx, JPy, JPz, BP = ps
    sin,cos,tan=np.sin,np.cos,np.tan

    dψ = σ3/cos(ϑ)
    dϑ = σ1
    dφ = σ2 - σ3*sin(ϑ)/cos(ϑ)
    dr = R*σ1 + σr
    dγ = -σ3*sin(ϑ)/cos(ϑ) + σg
    dxG = R*σ1*sin(ψ)*cos(ϑ) + R*σ2*cos(ψ)
    dyG = -R*σ1*cos(ψ)*cos(ϑ) + R*σ2*sin(ψ)

    ret = np.array([dψ,dϑ,dφ,dr,dγ,dxG,dyG])
    return ret

@jit
def dqs2sigmas(dqs,qs,ps):
    dψ,dϑ,dφ,dr,dγ,dxG,dyG = dqs
    ψ, ϑ, φ, r, γ, xG, yG  =  qs
    g, R, mw, JW1, JW2, mr, JR, BR, h, mp, JPx, JPy, JPz, BP = ps
    sin,cos,tan=np.sin,np.cos,np.tan

    σ1 = dϑ
    σ2 = dφ + dψ*sin(ϑ)
    σ3 = dψ*cos(ϑ)
    σr = -R*dϑ + dr
    σg = dγ + dψ*sin(ϑ)

    sigmas = np.array([σ1,σ2,σ3,σr,σg])
    return sigmas 

@jit
def rhs(t,xs,ps): # right hand side for simulation, dynamics : M\(-C[:5]), kinematics: -C
    M   = Mfcn(t,xs,ps)
    us  = control(t,xs,ps)
    ret = -Cfcn(t,xs,ps,us)
    ret[:5] = np.linalg.solve(M,ret[:5])
    return ret

#          | WHEEL                                     | LATERAL ROD           | PENDULUM                                                      |
#      g,  | R,     mw,    JW1 (y),       JW2 (x&z),   | mr,    JR (x&z),   BR,| h,     mp,    JPx,           JPy,           JPz            BP |
psn = [9.81, 0.253, 2.436, 0.09099921839, 0.04591427768, 0.262, 0.00159165, 0.0, 0.025, 2.799, 0.01290418213, 0.02090219895, 0.01118711607, 0.0]
psn = np.array(psn)

In [2]:
rand=np.random.random
tn=rand();xsn=rand(12);
usn = control(tn,xsn,psn)
Mfcn(tn,xsn,psn),Cfcn(tn,xsn,psn,usn),rhs(tn,xsn,psn)

(array([[ 0.47801833,  0.        , -0.01187237,  0.        ,  0.        ],
        [ 0.        ,  0.44285669, -0.02982537,  0.        ,  0.01444381],
        [-0.01187237, -0.02982537,  0.11289526,  0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.        ,  0.262     ,  0.        ],
        [ 0.        ,  0.01444381,  0.        ,  0.        ,  0.02265157]]),
 array([-3.80681759,  0.10182439,  0.18895762,  0.84049519, -0.36138631,
        -0.6391067 , -0.58977321, -0.20401385, -0.26270936, -0.59233711,
        -0.12375018,  0.126433  ]),
 array([ 7.93740343, -0.83916101, -1.06071912, -3.20799689, 16.48922918,
         0.6391067 ,  0.58977321,  0.20401385,  0.26270936,  0.59233711,
         0.12375018, -0.126433  ]))

In [5]:
%%timeit tn=rand();xsn=rand(12);psn=rand(14);usn=rand(2);

Mfcn(tn,xsn,psn)
Cfcn(tn,xsn,psn,usn)

1.7 μs ± 66.8 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [4]:
%%timeit tn=rand();xsn=rand(12);psn=rand(14);usn=rand(2);

rhs(tn,xsn,psn)

1.74 μs ± 40.3 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)
